# Decomposing Private Equity

In [43]:
import pandas as pd
import numpy as np

import statsmodels.api as sm

LOADFILE = 'private_equity_data.xlsx'
info = pd.read_excel(LOADFILE,sheet_name='descriptions').set_index('ticker')
rets = pd.read_excel(LOADFILE,sheet_name='total returns').set_index('date')

In [44]:
info    

,shortName,quoteType,currency,volume,totalAssets,longBusinessSummary
ticker,,,,,,
SPY,SPDR S&P 500,ETF,USD,65058380,6.727266e+11,The trust seeks to achieve its investment obje...
VTV,Vanguard Value ETF,ETF,USD,2008018,2.069965e+11,The manager employs an indexing investment app...
AGG,iShares Core U.S. Aggregate Bon,ETF,USD,5268591,1.318272e+11,The index measures the performance of the tota...
HYG,iShares iBoxx $ High Yield Corp,ETF,USD,35079406,1.854291e+10,The underlying index is a rules-based index co...
IEF,iShares 7-10 Year Treasury Bond,ETF,USD,5154346,3.851085e+10,The underlying index measures the performance ...
PSP,Invesco Global Listed Private E,ETF,USD,46427,3.347804e+08,The fund generally will invest at least 90% of...
PEX,ProShares Global Listed Private,ETF,USD,4331,1.390970e+07,The fund invests in financial instruments that...
APO,"Apollo Global Management, Inc.",EQUITY,USD,3430863,NaN,"Apollo Global Management, Inc. is a private eq..."
BX,Blackstone Inc.,EQUITY,USD,3218280,NaN,Blackstone Inc. is an alternative asset manage...


In [45]:
rets.head()

,SPY,VTV,AGG,HYG,IEF,PSP,PEX,APO,BX,KKR,CG,SHV
date,,,,,,,,,,,,
2015-01-09,-0.005744,-0.007218,0.005433,0.005580,0.013517,-0.007306,-0.006087,0.012728,0.002344,0.010643,-0.056778,0.000181
2015-01-16,-0.012827,-0.011681,0.004233,-0.005882,0.010188,-0.013799,-0.010972,0.003351,-0.015785,-0.033277,-0.064334,0.000181
2015-01-23,0.016565,0.011939,0.001883,0.004912,0.001559,0.019590,0.009288,0.027975,0.070983,0.057081,0.031765,0.000000
2015-01-30,-0.026931,-0.034323,0.005998,0.002444,0.011992,-0.000915,0.008947,0.010154,0.035496,-0.010305,0.024942,-0.000091
2015-02-06,0.030584,0.037887,-0.011746,0.007869,-0.024333,0.027473,0.028883,-0.022919,-0.033744,0.029571,0.011787,0.000091


In [46]:
excess_rets = rets.sub(rets["SHV"], axis=0).drop(columns="SHV")
excess_rets.head()

,SPY,VTV,AGG,HYG,IEF,PSP,PEX,APO,BX,KKR,CG
date,,,,,,,,,,,
2015-01-09,-0.005925,-0.007399,0.005252,0.005399,0.013336,-0.007487,-0.006268,0.012547,0.002163,0.010461,-0.056959
2015-01-16,-0.013009,-0.011862,0.004052,-0.006064,0.010007,-0.013981,-0.011154,0.003170,-0.015966,-0.033459,-0.064515
2015-01-23,0.016565,0.011939,0.001883,0.004912,0.001559,0.019590,0.009288,0.027975,0.070983,0.057081,0.031765
2015-01-30,-0.026840,-0.034232,0.006088,0.002535,0.012083,-0.000824,0.009038,0.010245,0.035587,-0.010214,0.025032
2015-02-06,0.030493,0.037797,-0.011837,0.007778,-0.024424,0.027382,0.028792,-0.023010,-0.033835,0.029480,0.011696


# 1. Performance

Calculate performance stats for the assets, both stand-alone (mean, vol, Sharpe) as well as linear decompositions (alpha, beta, info ratio, r-squared).

In [47]:
# mean, vol, sharpe

means = excess_rets.mean() * 52

vols = excess_rets.std() * np.sqrt(52)

sharpe = means/vols

stand_alone_stats = pd.DataFrame({"Mean": means, "Volatility (ann.)": vols, "Sharpe": sharpe})

stand_alone_stats

,Mean,Volatility (ann.),Sharpe
SPY,0.123097,0.169669,0.725516
VTV,0.086451,0.164095,0.526835
AGG,-0.007880,0.050925,-0.154742
HYG,-0.004747,0.095436,-0.049739
IEF,-0.007308,0.064357,-0.113559
PSP,0.049428,0.231667,0.213358
PEX,-0.014267,0.214613,-0.066479
APO,0.231547,0.367902,0.629371
BX,0.217569,0.358169,0.607448
KKR,0.228559,0.364961,0.626255


In [48]:
ann = 52
results = {}
for asset in excess_rets.columns.drop("SPY"):          
    X = sm.add_constant(excess_rets["SPY"])
    model = sm.OLS(excess_rets[asset], X, missing="drop").fit()

    alpha = model.params["const"] * ann            # annualized
    TE    = model.resid.std() * np.sqrt(ann)       # annualized tracking error

    results[asset] = {"alpha (ann.)": alpha,
                      "beta":  model.params["SPY"],
                      "r_squared": model.rsquared, "Tracking Error": TE, "info-ratio": alpha/TE}

results = pd.DataFrame(results).T
results

,alpha (ann.),beta,r_squared,Tracking Error,info-ratio
VTV,-0.023904,0.896488,0.859217,0.061570,-0.388247
AGG,-0.015953,0.065578,0.047737,0.049694,-0.321015
HYG,-0.056986,0.424370,0.569210,0.062639,-0.909751
IEF,-0.003964,-0.027171,0.005131,0.064192,-0.061747
PSP,-0.095972,1.181182,0.748358,0.116213,-0.825830
PEX,-0.134605,0.977579,0.597300,0.136191,-0.988354
APO,0.049338,1.480204,0.465995,0.268847,0.183517
BX,0.037408,1.463572,0.480678,0.258111,0.144928
KKR,0.034763,1.574329,0.535674,0.248690,0.139785
CG,-0.035943,1.466172,0.459224,0.269950,-0.133147


# 2. Replication

Can we replicate private equity APO using the following?

- public equity SPY

- treasury bonds IEF

- high-yield corp bonds HYG



In [49]:
X = excess_rets[["SPY", "IEF", "HYG"]]
model = sm.OLS(excess_rets["APO"], X, missing="drop").fit()
print(model.params)
print("R-squared: ", model.rsquared)
weights = model.params

repl_port_ret = X @ weights
repl_port_ret

SPY    1.086206
IEF   -0.698868
HYG    0.902516
dtype: float64
R-squared:  0.4989500095571676


date
2015-01-09   -0.010884
2015-01-16   -0.026596
2015-01-23    0.021337
2015-01-30   -0.035310
2015-02-06    0.057211
                ...   
2025-09-05    0.002255
2025-09-12    0.015808
2025-09-19    0.015090
2025-09-26   -0.002929
2025-10-03    0.002461
Length: 561, dtype: float64

In [50]:
# mean, vol, sharpe

repl_mean = repl_port_ret.mean() * 52

repl_vol = repl_port_ret.std() * np.sqrt(52)

repl_sharpe = repl_mean/repl_vol

print("Mean: ", repl_mean)
print("Volatility: ", repl_vol)
print("Sharpe: ", repl_sharpe)

Mean:  0.1345323380173799
Volatility:  0.26019314351514095
Sharpe:  0.517047975207507


_Answer:_ Regressing APO on SPY, IEF and HYG without an intercept (since an alpha cannot be bought), the replicating portfolio earns an annualized excess return of 13.5% vs. 23.2% for APO, with a Sharpe ratio of 0.52 vs. 0.63. The factors capture APO's systematic exposure: long equity (1.09 SPY) and long credit spread (0.90 HYG vs. -0.70 IEF), with the weights summing to 1.29 and thus financed by borrowing at the risk-free rate. However, around 10% per year of APO's return cannot be replicated with liquid public assets.

# How does that compare to APO against SPY alone?

_Answer:_ Compared to APO against SPY alone, adding IEF and HYG barely improves the fit (R-squared of 0.497 vs. 0.466). It is important to note, though, that the SPY-only regression with an intercept attributes an annualized alpha of 4.9% to APO, but with a tracking error of 26.9%. This means that the information ratio is only 0.18, i.e. the alpha is small relative to the unhedgeable idiosyncratic risk. This alpha is also what, as mentioned above, no combination of the factors can reproduce.

# Replication with intercept (curiosity purposes)

In [51]:
X = sm.add_constant(excess_rets[["SPY", "IEF", "HYG"]])
model = sm.OLS(excess_rets["APO"], X, missing="drop").fit()
print(model.params)
print("R-squared: ", model.rsquared)
weights = model.params

repl_port_ret = X @ weights
repl_port_ret

const    0.001915
SPY      1.066000
IEF     -0.703990
HYG      0.930969
dtype: float64
R-squared:  0.49650177514561467


date
2015-01-09   -0.008763
2015-01-16   -0.024642
2015-01-23    0.023050
2015-01-30   -0.032842
2015-02-06    0.058857
                ...   
2025-09-05    0.004080
2025-09-12    0.017425
2025-09-19    0.016933
2025-09-26   -0.000999
2025-10-03    0.004353
Length: 561, dtype: float64

In [52]:
# mean, vol, sharpe

repl_mean = repl_port_ret.mean() * 52

repl_vol = repl_port_ret.std() * np.sqrt(52)

repl_sharpe = repl_mean/repl_vol

print("Mean: ", repl_mean)
print("Volatility: ", repl_vol)
print("Sharpe: ", repl_sharpe)

Mean:  0.23154697772883473
Volatility:  0.25923429919548546
Sharpe:  0.8931957632436128


_Note:_ With an intercept, the fitted values include the alpha (0.0019 weekly, c. 10% annualized), so the "replication" matches APO's mean by construction and shows a Sharpe of 0.89. Since the alpha cannot be traded, this is not an achievable replication; the ~10% alpha is exactly the gap between the no-intercept replication (13.5%) and APO (23.2%).